<a href="https://colab.research.google.com/github/eusmani/ML-basics/blob/main/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Data source: Telco Customer Churn Dataset
# https://github.com/MatthewChatham/ames/blob/master/train.csv


# Generate sample customer data
# Load the dataset
url = "https://raw.githubusercontent.com/KimathiNewton/Telco-Customer-Churn/master/Datasets/telco_churn.csv"
df = pd.read_csv(url)

# Clean TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Clean and convert Churn values
df["Churn"] = (
    df["Churn"]
    .astype(str)
    .str.strip()
    .map({
        "Yes": 1,
        "No": 0,
        "1": 1,
        "0": 0
    })
)

# Remove missing values after all conversions
df = df.dropna(subset=[
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "Churn"
])

print("Churn values:", df["Churn"].unique())
print("Number of customer records:", len(df))

print("Number of customer records:", len(df))
print(df.head())

# Select realistic customer features
features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract",
    "InternetService",
    "PaymentMethod"
]

X = df[features]
y = df["Churn"]

# Numerical and categorical columns
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "Contract",
    "InternetService",
    "PaymentMethod"
]

# Scale numerical features and encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Create the logistic regression pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
predictions = model.predict(X_test)

print("\nModel Accuracy:")
print("{:.2f}%".format(accuracy_score(y_test, predictions) * 100))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    "tenure": [12],
    "MonthlyCharges": [85.50],
    "TotalCharges": [1026.00],
    "Contract": ["Month-to-month"],
    "InternetService": ["Fiber optic"],
    "PaymentMethod": ["Electronic check"]
})

churn_probability = model.predict_proba(new_customer)[0][1]

# Classify customer using a 0.5 threshold
threshold = 0.5
churn_prediction = 1 if churn_probability >= threshold else 0

print("\nNew Customer Prediction:")
print(f"Churn Probability: {churn_probability:.2f}")
print(f"Churn Prediction: {churn_prediction}")

if churn_prediction == 1:
    print("This customer is at risk of churning.")
else:
    print("This customer is not currently classified as at risk.")

# Display model coefficients
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = model.named_steps[
    "classifier"
].coef_[0]

print("\nModel Coefficients:")
for feature, coefficient in zip(feature_names, coefficients):
    print(f"{feature}: {coefficient:.2f}")

Churn values: [0. 1.]
Number of customer records: 2040
Number of customer records: 2040
      Unnamed: 0  customerID  gender SeniorCitizen Partner Dependents  tenure  \
3000           0  5600-PDUJF    Male             0      No         No       6   
3001           1  8292-TYSPY    Male             0      No         No      19   
3002           2  0567-XRHCU  Female             0     Yes        Yes      69   
3003           3  1867-BDVFH    Male             0     Yes        Yes      11   
3004           4  2067-QYTCF  Female             0     Yes         No      64   

     PhoneService     MultipleLines InternetService  ... DeviceProtection  \
3000          Yes                No             DSL  ...               No   
3001          Yes                No             DSL  ...              Yes   
3002           No  No phone service             DSL  ...              Yes   
3003          Yes               Yes     Fiber optic  ...               No   
3004          Yes               Yes     